# Quantum Autoencoder: DIFE and LS-SWAP Implementation

**Implementation of two new quantum autoencoder strategies for fraud detection:**

1. **DIFE (Destructive Interference Fidelity Estimation)**: Ancilla-free QAE using compute/uncompute sequence
2. **LS-SWAP (Latent Space SWAP Test)**: Resource-optimized SWAP test on latent space only

**Based on**: Technical specification for advanced QAE strategies with reduced resource requirements

---

## Key Features:

### DIFE Strategy:
- **Ancilla-free**: No additional reference or control qubits needed
- **Compute/Uncompute**: Forward pass followed by adjoint operation
- **Destructive Interference**: Measures probability of returning to |0⟩ state
- **8 qubits total** (same as enhanced_qvae data qubits)

### LS-SWAP Strategy:
- **Latent Space Focus**: SWAP test only on compressed representation
- **Resource Efficient**: Fewer ancilla qubits than full SWAP test
- **11 qubits total** (8 data + 2 reference + 1 control)
- **Maintains SWAP test benefits** with reduced complexity

**Goal**: Evaluate if these resource-optimized strategies can match or exceed enhanced_qvae performance while using fewer quantum resources.

In [1]:
# Core libraries for data processing and machine learning
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)
from sklearn.decomposition import PCA
import time

# Quantum machine learning framework
import pennylane as qml
import pennylane.numpy as pnp

print("Libraries imported successfully!")
print(f"PennyLane version: {qml.__version__}")
print("Ready for DIFE and LS-SWAP implementation!")

c:\ProgramData\anaconda3\envs\my312\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Libraries imported successfully!
PennyLane version: 0.41.1
Ready for DIFE and LS-SWAP implementation!


In [2]:
# ==========================================
# Data Loading and Preprocessing Pipeline
# ==========================================

# Load preprocessed credit card fraud dataset
df = pd.read_csv("../../data/preprocessed-creditcard.csv")
X = df.drop("Class", axis=1).values  # Feature matrix
y = df["Class"].values                # Target labels (0: normal, 1: fraud)

print(f"Dataset loaded: {X.shape[0]} samples, {X.shape[1]} features")
print(f"Fraud rate: {np.mean(y):.4f} ({np.sum(y)} fraud cases)")

# Stratified train-test split to maintain class distribution
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Feature standardization using Z-score normalization
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

# Dimensionality reduction using PCA
# 4D PCA for parallel embedding (4 features → 8 qubits with 2x embedding)
pca_4d = PCA(n_components=4, random_state=42)
X_train_4d = pca_4d.fit_transform(X_train)
X_test_4d  = pca_4d.transform(X_test)

# 8D PCA for direct embedding (8 features → 8 qubits with 1x embedding)
pca_8d = PCA(n_components=8, random_state=42)
X_train_8d = pca_8d.fit_transform(X_train)
X_test_8d  = pca_8d.transform(X_test)

print(f"\nTraining set shapes:")
print(f"  4D PCA: {X_train_4d.shape} (for parallel embedding)")
print(f"  8D PCA: {X_train_8d.shape} (for direct embedding)")
print(f"Test set shapes:")
print(f"  4D PCA: {X_test_4d.shape} (for parallel embedding)")
print(f"  8D PCA: {X_test_8d.shape} (for direct embedding)")

print(f"\n4D PCA explained variance ratio: {pca_4d.explained_variance_ratio_}")
print(f"4D Total variance explained: {np.sum(pca_4d.explained_variance_ratio_):.4f}")
print(f"\n8D PCA explained variance ratio: {pca_8d.explained_variance_ratio_}")
print(f"8D Total variance explained: {np.sum(pca_8d.explained_variance_ratio_):.4f}")

Dataset loaded: 946 samples, 30 features
Fraud rate: 0.5000 (473 fraud cases)

Training set shapes:
  4D PCA: (756, 4) (for parallel embedding)
  8D PCA: (756, 8) (for direct embedding)
Test set shapes:
  4D PCA: (190, 4) (for parallel embedding)
  8D PCA: (190, 8) (for direct embedding)

4D PCA explained variance ratio: [0.38421646 0.10954544 0.06067923 0.05752846]
4D Total variance explained: 0.6120

8D PCA explained variance ratio: [0.38421646 0.10954544 0.06067923 0.05752846 0.04385476 0.03912248
 0.03625876 0.03386247]
8D Total variance explained: 0.7651


In [3]:
# ==========================================
# Configuration for DIFE and LS-SWAP Strategies
# ==========================================

# EXPERIMENT CONFIGURATIONS
# We will test 4 different configurations:
# 1. DIFE + 4D PCA + Parallel Embedding (2x)
# 2. DIFE + 8D PCA + Direct Embedding (1x) 
# 3. LS-SWAP + 4D PCA + Parallel Embedding (2x)
# 4. LS-SWAP + 8D PCA + Direct Embedding (1x)

EXPERIMENT_CONFIGS = {
    'parallel': {
        'use_parallel_embedding': True,
        'parallel_factor': 2,
        'input_features': 4,
        'data_qubits': 8,  # 4 features × 2x parallel
        'data_key': '4d',
        'description': 'Parallel Embedding (4D→8Q)'
    },
    'direct': {
        'use_parallel_embedding': False, 
        'parallel_factor': 1,
        'input_features': 8,
        'data_qubits': 8,  # 8 features × 1x direct
        'data_key': '8d',
        'description': 'Direct Embedding (8D→8Q)'
    }
}

# SHARED ENHANCED qVAE FEATURES
USE_DATA_REUPLOADING = True     # Embed data at each variational layer
USE_ALTERNATE_EMBEDDING = True  # Alternate between RY and RX rotations

# LS-SWAP Configuration  
LS_SWAP_LATENT_QUBITS = 2        # Number of latent space qubits for SWAP test

# QUANTUM ARCHITECTURE PARAMETERS
L = 4  # Number of variational layers

# TRAINING CONFIGURATION
TRAINING_CONFIG = {
    'epochs': 100,          # Updated for more thorough training
    'batch_size': 16,       # Batch size for all experiments
    'learning_rate': 0.001  # Adam optimizer stepsize
}

print("="*80)
print("QUANTUM AUTOENCODER - COMPREHENSIVE EXPERIMENT DESIGN")
print("="*80)
print("Experiment Configurations:")
for config_name, config in EXPERIMENT_CONFIGS.items():
    print(f"\n{config_name.upper()} EMBEDDING:")
    print(f"  - Description: {config['description']}")
    print(f"  - Input Features: {config['input_features']}D")
    print(f"  - Data Qubits: {config['data_qubits']}")
    print(f"  - Parallel Factor: {config['parallel_factor']}x")

print(f"\nShared Features:")
print(f"  - Data Re-uploading: {USE_DATA_REUPLOADING}")
print(f"  - Alternate RY/RX: {USE_ALTERNATE_EMBEDDING}")
print(f"  - Variational Layers: {L}")
print(f"  - LS-SWAP Latent Qubits: {LS_SWAP_LATENT_QUBITS}")

print(f"\nTraining Configuration: {TRAINING_CONFIG}")

print(f"\n4 TOTAL EXPERIMENTS:")
print(f"  1. DIFE + Parallel Embedding (4D→8Q)")
print(f"  2. DIFE + Direct Embedding (8D→8Q)")
print(f"  3. LS-SWAP + Parallel Embedding (4D→8Q)")
print(f"  4. LS-SWAP + Direct Embedding (8D→8Q)")
print("="*80)

QUANTUM AUTOENCODER - COMPREHENSIVE EXPERIMENT DESIGN
Experiment Configurations:

PARALLEL EMBEDDING:
  - Description: Parallel Embedding (4D→8Q)
  - Input Features: 4D
  - Data Qubits: 8
  - Parallel Factor: 2x

DIRECT EMBEDDING:
  - Description: Direct Embedding (8D→8Q)
  - Input Features: 8D
  - Data Qubits: 8
  - Parallel Factor: 1x

Shared Features:
  - Data Re-uploading: True
  - Alternate RY/RX: True
  - Variational Layers: 4
  - LS-SWAP Latent Qubits: 2

Training Configuration: {'epochs': 100, 'batch_size': 16, 'learning_rate': 0.001}

4 TOTAL EXPERIMENTS:
  1. DIFE + Parallel Embedding (4D→8Q)
  2. DIFE + Direct Embedding (8D→8Q)
  3. LS-SWAP + Parallel Embedding (4D→8Q)
  4. LS-SWAP + Direct Embedding (8D→8Q)


In [4]:
# ==========================================
# Shared Quantum Circuit Functions
# ==========================================

def enhanced_qvae_layer(inputs, weights, layer_idx, n_layers, n_qubits, 
                       embedding_config, reupload=True, alternate_embedding=False):
    """
    Enhanced qVAE layer with configurable embedding (parallel or direct).
    
    Args:
        inputs: Input data features
        weights: Trainable parameters for this layer
        layer_idx: Current layer index
        n_layers: Total number of layers
        n_qubits: Number of data qubits
        embedding_config: Configuration dict with embedding settings
        reupload: Whether to use data re-uploading
        alternate_embedding: Whether to alternate between RY and RX
    """
    use_parallel = embedding_config['use_parallel_embedding']
    parallel_factor = embedding_config['parallel_factor']
    
    # Data embedding (with re-uploading if enabled)
    if not reupload or layer_idx == 0:  # Always embed on first layer
        for i, feature in enumerate(inputs):
            if use_parallel:
                # Parallel embedding: replicate data across multiple qubits
                for p in range(parallel_factor):
                    qubit_idx = i * parallel_factor + p
                    if qubit_idx < n_qubits:
                        if alternate_embedding and (i + p) % 2 == 1:
                            qml.RX(feature, wires=qubit_idx)
                        else:
                            qml.RY(feature, wires=qubit_idx)
            else:
                # Direct embedding: one feature per qubit
                if i < n_qubits:
                    if alternate_embedding and i % 2 == 1:
                        qml.RX(feature, wires=i)
                    else:
                        qml.RY(feature, wires=i)
    
    # Parameterized rotations for each qubit
    for w in range(n_qubits):
        qml.RY(weights[w, 0], wires=w)
        qml.RZ(weights[w, 1], wires=w)
    
    # Entangling gates with periodic boundary
    if n_qubits > 1:
        for w in range(n_qubits):
            control = w
            target = (w + 1) % n_qubits
            qml.CNOT(wires=[control, target])
    
    # Data re-uploading for intermediate layers
    if reupload and layer_idx < n_layers - 1:
        for i, feature in enumerate(inputs):
            if use_parallel:
                # Parallel embedding: replicate data across multiple qubits
                for p in range(parallel_factor):
                    qubit_idx = i * parallel_factor + p
                    if qubit_idx < n_qubits:
                        if alternate_embedding and (i + p) % 2 == 1:
                            qml.RX(feature, wires=qubit_idx)
                        else:
                            qml.RY(feature, wires=qubit_idx)
            else:
                # Direct embedding: one feature per qubit
                if i < n_qubits:
                    if alternate_embedding and i % 2 == 1:
                        qml.RX(feature, wires=i)
                    else:
                        qml.RY(feature, wires=i)

def encoder_ansatz(x, weights, n_qubits, embedding_config):
    """
    Complete encoder ansatz for use in both DIFE and LS-SWAP strategies.
    Now supports both parallel and direct embedding modes.
    
    Args:
        x: Input features
        weights: Trainable parameters
        n_qubits: Number of data qubits
        embedding_config: Configuration dict with embedding settings
    """
    # Apply enhanced qVAE layers with configurable embedding
    for l in range(L):
        enhanced_qvae_layer(
            inputs=x,
            weights=weights[l],
            layer_idx=l,
            n_layers=L,
            n_qubits=n_qubits,
            embedding_config=embedding_config,
            reupload=USE_DATA_REUPLOADING,
            alternate_embedding=USE_ALTERNATE_EMBEDDING
        )

def data_embedding_layer(x, embedding_config, n_qubits, alternate_embedding=False):
    """
    Standalone data embedding function for use in DIFE circuits.
    
    Args:
        x: Input features
        embedding_config: Configuration dict with embedding settings
        n_qubits: Number of data qubits
        alternate_embedding: Whether to alternate between RY and RX
    """
    use_parallel = embedding_config['use_parallel_embedding']
    parallel_factor = embedding_config['parallel_factor']
    
    for i, feature in enumerate(x):
        if use_parallel:
            # Parallel embedding: replicate data across multiple qubits
            for p in range(parallel_factor):
                qubit_idx = i * parallel_factor + p
                if qubit_idx < n_qubits:
                    if alternate_embedding and (i + p) % 2 == 1:
                        qml.RX(feature, wires=qubit_idx)
                    else:
                        qml.RY(feature, wires=qubit_idx)
        else:
            # Direct embedding: one feature per qubit
            if i < n_qubits:
                if alternate_embedding and i % 2 == 1:
                    qml.RX(feature, wires=i)
                else:
                    qml.RY(feature, wires=i)

print("Shared quantum circuit functions defined successfully!")
print("  - enhanced_qvae_layer: Configurable parallel/direct embedding")
print("  - encoder_ansatz: Complete encoder with embedding config")
print("  - data_embedding_layer: Standalone embedding function")

Shared quantum circuit functions defined successfully!
  - enhanced_qvae_layer: Configurable parallel/direct embedding
  - encoder_ansatz: Complete encoder with embedding config
  - data_embedding_layer: Standalone embedding function


In [5]:
# ==========================================
# DIFE Strategy Implementation
# ==========================================

def dife_circuit(x, weights, n_qubits, embedding_config):
    """
    DIFE (Destructive Interference Fidelity Estimation) circuit implementation.
    
    Now supports both parallel and direct embedding configurations.
    
    Circuit sequence:
    1. Encode input data into initial state |ψ(x)⟩
    2. Apply encoder transformation (compression)
    3. Apply partial decoder transformation (asymmetric reconstruction)
    4. Measure single observable that reflects reconstruction quality
    
    Args:
        x: Input data features
        weights: Trainable parameters
        n_qubits: Number of data qubits (8)
        embedding_config: Configuration dict with embedding settings
    
    Returns:
        Single scalar reconstruction fidelity score
    """
    # Step 1: Encode input data to create reference state |ψ(x)⟩
    data_embedding_layer(x, embedding_config, n_qubits, USE_ALTERNATE_EMBEDDING)
    
    # Step 2: Apply encoder transformation (compression phase)
    for l in range(L):
        # Parameterized rotations
        for w in range(n_qubits):
            qml.RY(weights[l][w, 0], wires=w)
            qml.RZ(weights[l][w, 1], wires=w)
        
        # Entangling gates
        if n_qubits > 1:
            for w in range(n_qubits):
                control = w
                target = (w + 1) % n_qubits
                qml.CNOT(wires=[control, target])
        
        # Data re-uploading for intermediate layers (if enabled)
        if USE_DATA_REUPLOADING and l < L - 1:
            data_embedding_layer(x, embedding_config, n_qubits, USE_ALTERNATE_EMBEDDING)
    
    # Step 3: Apply partial decoder transformation (asymmetric reconstruction)
    # Only reverse the last layer to create imperfect reconstruction
    l = L - 1  # Last layer only
    
    # Reverse entangling gates for last layer
    if n_qubits > 1:
        for w in reversed(range(n_qubits)):
            control = w
            target = (w + 1) % n_qubits
            qml.CNOT(wires=[control, target])
    
    # Reverse parameterized rotations for last layer
    for w in reversed(range(n_qubits)):
        qml.RZ(-weights[l][w, 1], wires=w)
        qml.RY(-weights[l][w, 0], wires=w)
    
    # Step 4: Single measurement that reflects reconstruction quality
    # Use a simple Pauli-Z measurement on the first qubit
    return qml.expval(qml.PauliZ(0))

print("DIFE Strategy Implementation Complete!")
print("  - Supports both parallel and direct embedding")
print("  - Ancilla-free: Uses only 8 data qubits")
print("  - Partial reconstruction: Encoder + last layer decoder only")
print("  - Single measurement: PauliZ on first qubit")

DIFE Strategy Implementation Complete!
  - Supports both parallel and direct embedding
  - Ancilla-free: Uses only 8 data qubits
  - Partial reconstruction: Encoder + last layer decoder only
  - Single measurement: PauliZ on first qubit


In [6]:
# ==========================================
# LS-SWAP Strategy Implementation 
# ==========================================

def latent_space_swap_test(n_data_qubits, n_latent, total_qubits):
    """
    Implement SWAP test restricted to latent space qubits only.
    
    This resource-optimized SWAP test operates on the assumption that 
    the fidelity of the compressed latent state is a sufficient proxy 
    for the fidelity of the entire system.
    
    Qubit layout:
    - Latent qubits: wires 0 to n_latent-1 (first 2 data qubits)
    - Reference qubits: wires n_data_qubits to n_data_qubits+n_latent-1
    - Control qubit: wire total_qubits-1 (last qubit)
    
    Args:
        n_data_qubits: Number of data qubits (8)
        n_latent: Number of latent space qubits (2)
        total_qubits: Total qubits in circuit (11)
    
    Returns:
        Expectation value of PauliZ on control qubit
    """
    control_qubit = total_qubits - 1  # Last qubit as control
    
    # Apply Hadamard to control qubit
    qml.Hadamard(wires=control_qubit)
    
    # Controlled SWAP operations between latent and reference qubits
    for i in range(n_latent):
        latent_qubit = i                           # Latent space qubit (0, 1)
        reference_qubit = n_data_qubits + i        # Reference qubit (8, 9)
        qml.CSWAP(wires=[control_qubit, latent_qubit, reference_qubit])
    
    # Final Hadamard on control qubit  
    qml.Hadamard(wires=control_qubit)
    
    # Measure control qubit
    return qml.expval(qml.PauliZ(control_qubit))

def ls_swap_circuit(x, weights, n_qubits, total_qubits, embedding_config):
    """
    LS-SWAP (Latent Space SWAP Test) circuit implementation.
    
    Now supports both parallel and direct embedding configurations.
    
    Circuit sequence:
    1. Apply enhanced qVAE encoder to data qubits
    2. Perform SWAP test between latent qubits (0,1) and reference qubits (8,9)
    3. Measure control qubit for fidelity estimation
    
    Args:
        x: Input data features  
        weights: Trainable parameters
        n_qubits: Number of data qubits (8)
        total_qubits: Total qubits including ancilla (11)
        embedding_config: Configuration dict with embedding settings
        
    Returns:
        SWAP test expectation value for fidelity estimation
    """
    # Apply enhanced qVAE encoder to data qubits
    encoder_ansatz(x, weights, n_qubits, embedding_config)
    
    # Perform latent space SWAP test
    return latent_space_swap_test(n_qubits, LS_SWAP_LATENT_QUBITS, total_qubits)

print("LS-SWAP Strategy Implementation Complete!")
print("  - Supports both parallel and direct embedding")
print("  - Resource-efficient: 11 qubits vs 13 for full SWAP test")
print("  - Latent focus: SWAP test on compressed representation only")
print("  - Wire layout: Latent (0,1), Data (2-7), Reference (8,9), Control (10)")

LS-SWAP Strategy Implementation Complete!
  - Supports both parallel and direct embedding
  - Resource-efficient: 11 qubits vs 13 for full SWAP test
  - Latent focus: SWAP test on compressed representation only
  - Wire layout: Latent (0,1), Data (2-7), Reference (8,9), Control (10)


In [7]:
# ==========================================
# Comprehensive Training Functions
# ==========================================

def compute_batch_cost(samples, circuit, weights, strategy_name):
    """
    Compute batch cost for different strategies with appropriate loss functions.
    
    Args:
        samples: Batch data samples
        circuit: Quantum circuit function
        weights: Trainable parameters
        strategy_name: 'dife' or 'ls_swap' for strategy-specific handling
        
    Returns:
        linear_loss: Linear loss (1 - fidelity)
        squared_loss: Squared loss (1 - fidelity)^2
    """
    linear_errors = []
    squared_errors = []
    
    for sample in samples:
        features = pnp.array(sample, requires_grad=False)
        expval = circuit(features, weights)
        
        # Strategy-specific fidelity calculation
        if strategy_name == 'dife':
            # DIFE: expval is already the fidelity measurement
            fidelity = expval
        elif strategy_name == 'ls_swap':
            # LS-SWAP: Convert SWAP test expval to fidelity
            fidelity = (expval + 1.0) / 2.0
        else:
            raise ValueError(f"Unknown strategy: {strategy_name}")
        
        # Ensure fidelity is in valid range [0, 1]
        fidelity = pnp.clip(fidelity, 0.0, 1.0)
        
        # Linear and squared loss calculations
        linear_error = 1.0 - fidelity
        squared_error = linear_error ** 2
        
        linear_errors.append(linear_error)
        squared_errors.append(squared_error)
    
    linear_loss = pnp.mean(pnp.stack(linear_errors))
    squared_loss = pnp.mean(pnp.stack(squared_errors))
    
    return linear_loss, squared_loss

def train_single_experiment(strategy, embedding_type, data_dict):
    """
    Train a single experiment configuration.
    
    Args:
        strategy: 'dife' or 'ls_swap'
        embedding_type: 'parallel' or 'direct'
        data_dict: Dictionary containing train/test data
        
    Returns:
        Training results dictionary
    """
    experiment_name = f"{strategy}_{embedding_type}"
    embedding_config = EXPERIMENT_CONFIGS[embedding_type]
    
    print(f"\n{'='*70}")
    print(f"TRAINING: {experiment_name.upper()}")
    print(f"Strategy: {strategy.upper()}, Embedding: {embedding_config['description']}")
    print(f"{'='*70}")
    
    # Get training data
    data_key = embedding_config['data_key']
    X_train = data_dict[f'X_train_{data_key}']
    n_qubits = embedding_config['data_qubits']
    
    # Create quantum device
    if strategy == 'dife':
        total_qubits = n_qubits
        dev = qml.device("lightning.qubit", wires=total_qubits)
        
        @qml.qnode(dev)
        def circuit(x, weights):
            return dife_circuit(x, weights, n_qubits, embedding_config)
            
    elif strategy == 'ls_swap':
        total_qubits = n_qubits + LS_SWAP_LATENT_QUBITS + 1  # data + reference + control
        dev = qml.device("lightning.qubit", wires=total_qubits)
        
        @qml.qnode(dev)
        def circuit(x, weights):
            return ls_swap_circuit(x, weights, n_qubits, total_qubits, embedding_config)
    
    # Initialize weights
    weights = pnp.random.uniform(-pnp.pi, pnp.pi, (L, n_qubits, 2), requires_grad=True)
    
    # Training configuration
    epochs = TRAINING_CONFIG['epochs']
    batch_size = TRAINING_CONFIG['batch_size']
    optimizer = qml.AdamOptimizer(stepsize=TRAINING_CONFIG['learning_rate'])
    
    print(f"Configuration:")
    print(f"  - Strategy: {strategy.upper()}")
    print(f"  - Embedding: {embedding_config['description']}")
    print(f"  - Data Qubits: {n_qubits}")
    print(f"  - Total Qubits: {total_qubits}")
    print(f"  - Parameters: {np.prod(weights.shape)}")
    print(f"  - Training samples: {len(X_train)}")
    print(f"  - Epochs: {epochs}, Batch size: {batch_size}")
    
    # Training loop
    training_losses = []
    linear_losses = []
    squared_losses = []
    start_time = time.time()
    
    for epoch in range(epochs):
        epoch_linear_losses = []
        epoch_squared_losses = []
        
        for batch_start in range(0, len(X_train), batch_size):
            batch_end = min(batch_start + batch_size, len(X_train))
            X_batch = X_train[batch_start:batch_end]
            
            # Batch cost function wrapper
            def batch_cost_wrapper(w):
                linear_loss, squared_loss = compute_batch_cost(X_batch, circuit, w, strategy)
                return linear_loss
            
            # Optimize weights
            weights = optimizer.step(batch_cost_wrapper, weights)
            
            # Record losses
            linear_loss, squared_loss = compute_batch_cost(X_batch, circuit, weights, strategy)
            epoch_linear_losses.append(float(linear_loss))
            epoch_squared_losses.append(float(squared_loss))
        
        # Epoch summary
        avg_linear_loss = np.mean(epoch_linear_losses)
        avg_squared_loss = np.mean(epoch_squared_losses)
        linear_losses.append(avg_linear_loss)
        squared_losses.append(avg_squared_loss)
        training_losses.append(avg_linear_loss)
        
        if (epoch + 1) % 10 == 0 or epoch == epochs - 1:
            print(f"  Epoch {epoch+1:3d}/{epochs} - Linear Loss: {avg_linear_loss:.6f}, Squared Loss: {avg_squared_loss:.6f}")
    
    training_time = time.time() - start_time
    print(f"Training completed in {training_time:.1f}s - Final loss: {training_losses[-1]:.6f}")
    
    return {
        "experiment_name": experiment_name,
        "strategy": strategy,
        "embedding_type": embedding_type,
        "embedding_config": embedding_config,
        "weights": weights,
        "losses": training_losses,
        "linear_losses": linear_losses,
        "squared_losses": squared_losses,
        "circuit": circuit,
        "n_qubits": n_qubits,
        "total_qubits": total_qubits,
        "training_time": training_time,
        "final_loss": training_losses[-1],
    }

def train_all_experiments():
    """
    Train all 4 experiment combinations:
    1. DIFE + Parallel Embedding (4D→8Q)
    2. DIFE + Direct Embedding (8D→8Q) 
    3. LS-SWAP + Parallel Embedding (4D→8Q)
    4. LS-SWAP + Direct Embedding (8D→8Q)
    """
    print("STARTING COMPREHENSIVE TRAINING OF ALL EXPERIMENTS")
    print("="*80)
    
    # Prepare data dictionary
    data_dict = {
        'X_train_4d': X_train_4d,
        'X_test_4d': X_test_4d,
        'X_train_8d': X_train_8d,
        'X_test_8d': X_test_8d
    }
    
    results = {}
    total_start_time = time.time()
    
    # Define all experiments
    experiments = [
        ('dife', 'parallel'),
        ('dife', 'direct'),
        ('ls_swap', 'parallel'),
        ('ls_swap', 'direct')
    ]
    
    # Run each experiment
    for strategy, embedding_type in experiments:
        experiment_name = f"{strategy}_{embedding_type}"
        
        try:
            result = train_single_experiment(strategy, embedding_type, data_dict)
            results[experiment_name] = result
            print(f"✓ {experiment_name} completed successfully")
        except Exception as e:
            print(f"✗ {experiment_name} failed: {str(e)}")
            results[experiment_name] = {
                'experiment_name': experiment_name,
                'strategy': strategy, 
                'embedding_type': embedding_type,
                'error': str(e)
            }
    
    total_time = time.time() - total_start_time
    print(f"\n{'='*80}")
    print(f"ALL EXPERIMENTS COMPLETED IN {total_time:.1f}s")
    print(f"{'='*80}")
    
    # Training summary
    print(f"\nTRAINING SUMMARY:")
    print(f"{'Experiment':<20} {'Status':<10} {'Final Loss':<12} {'Time (s)':<10} {'Qubits':<8}")
    print(f"{'-'*70}")
    
    for experiment_name in ['dife_parallel', 'dife_direct', 'ls_swap_parallel', 'ls_swap_direct']:
        if experiment_name in results:
            result = results[experiment_name]
            if 'error' in result:
                print(f"{experiment_name:<20} {'FAILED':<10} {'N/A':<12} {'N/A':<10} {'N/A':<8}")
            else:
                print(f"{experiment_name:<20} {'SUCCESS':<10} {result['final_loss']:<12.6f} {result['training_time']:<10.1f} {result['total_qubits']:<8d}")
    
    print(f"\nExperiment Details:")
    print(f"  DIFE_PARALLEL: 4D PCA → 8 qubits with 2x parallel embedding")
    print(f"  DIFE_DIRECT: 8D PCA → 8 qubits with 1x direct embedding")
    print(f"  LS_SWAP_PARALLEL: 4D PCA → 8 qubits + 3 ancilla with 2x parallel embedding")
    print(f"  LS_SWAP_DIRECT: 8D PCA → 8 qubits + 3 ancilla with 1x direct embedding")
    print(f"\nReady for comprehensive evaluation and comparison!")
    
    return results

print("Comprehensive training functions defined successfully!")
print("  - train_single_experiment: Single configuration trainer")
print("  - train_all_experiments: All 4 combinations trainer")
print("  - compute_batch_cost: Strategy-specific loss computation")

Comprehensive training functions defined successfully!
  - train_single_experiment: Single configuration trainer
  - train_all_experiments: All 4 combinations trainer
  - compute_batch_cost: Strategy-specific loss computation


In [8]:
# ==========================================
# Execute All Experiments
# ==========================================

# Train all 4 experimental combinations
all_results = train_all_experiments()

STARTING COMPREHENSIVE TRAINING OF ALL EXPERIMENTS

TRAINING: DIFE_PARALLEL
Strategy: DIFE, Embedding: Parallel Embedding (4D→8Q)
Configuration:
  - Strategy: DIFE
  - Embedding: Parallel Embedding (4D→8Q)
  - Data Qubits: 8
  - Total Qubits: 8
  - Parameters: 64
  - Training samples: 756
  - Epochs: 100, Batch size: 16
  Epoch  10/100 - Linear Loss: 0.848847, Squared Loss: 0.749102
  Epoch  20/100 - Linear Loss: 0.638520, Squared Loss: 0.502060
  Epoch  30/100 - Linear Loss: 0.568026, Squared Loss: 0.450906
  Epoch  40/100 - Linear Loss: 0.537037, Squared Loss: 0.424614
  Epoch  50/100 - Linear Loss: 0.514766, Squared Loss: 0.402575
  Epoch  60/100 - Linear Loss: 0.500658, Squared Loss: 0.385506
  Epoch  70/100 - Linear Loss: 0.489760, Squared Loss: 0.372443
  Epoch  80/100 - Linear Loss: 0.479044, Squared Loss: 0.361901
  Epoch  90/100 - Linear Loss: 0.463764, Squared Loss: 0.348880
  Epoch 100/100 - Linear Loss: 0.446305, Squared Loss: 0.334389
Training completed in 2346.6s - Final 

In [9]:
# ==========================================
# Comprehensive Evaluation Functions
# ==========================================

def compute_metrics(y_true, y_pred):
    """
    Compute comprehensive evaluation metrics for binary classification.
    
    Includes standard metrics plus G-Mean which is particularly important
    for imbalanced datasets as it balances sensitivity and specificity.
    """
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    
    # Standard classification metrics
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)  # Sensitivity
    f1   = f1_score(y_true, y_pred, zero_division=0)
    
    # Specificity (True Negative Rate)
    spec = tn / (tn + fp) if (tn + fp) else 0.
    
    # Geometric Mean of Sensitivity and Specificity
    # Balanced metric for imbalanced datasets
    gmean = (rec * spec) ** 0.5
    
    return dict(TN=tn, FP=fp, FN=fn, TP=tp,
                Accuracy=acc, Precision=prec,
                Recall=rec, F1=f1, Specificity=spec, Gmean=gmean)

def evaluate_single_experiment(experiment_name, result_data, y_test):
    """
    Evaluate a single experiment on test data.
    """
    print(f"\n{'='*80}")
    print(f"EVALUATING: {experiment_name.upper()}")
    print(f"{'='*80}")
    
    if 'error' in result_data:
        print(f"Experiment failed during training: {result_data['error']}")
        return None
    
    # Extract experiment details
    strategy = result_data['strategy']
    embedding_type = result_data['embedding_type']
    embedding_config = result_data['embedding_config']
    weights = result_data['weights']
    circuit = result_data['circuit']
    
    # Get appropriate test data
    data_key = embedding_config['data_key']
    if data_key == '4d':
        X_test = X_test_4d
    else:  # data_key == '8d'
        X_test = X_test_8d
    
    print(f"Strategy: {strategy.upper()}")
    print(f"Embedding: {embedding_config['description']}")
    print(f"Test data: {X_test.shape}")
    print(f"Computing reconstruction fidelities for {len(X_test)} test samples...")
    
    # Compute fidelities
    fidelities = []
    for i, x in enumerate(X_test):
        if i % 200 == 0:
            print(f"  Processed {i}/{len(X_test)} samples")
        
        features = pnp.array(x, requires_grad=False)
        
        try:
            # Use the trained circuit for this experiment
            expval = circuit(features, weights)
            
            # Strategy-specific fidelity calculation
            if strategy == "dife":
                # DIFE: expval is already the fidelity measurement
                fidelity = float(expval)
            elif strategy == "ls_swap":
                # LS-SWAP: Convert SWAP test expval to fidelity
                fidelity = float((expval + 1.0) / 2.0)
            else:
                fidelity = 0.5  # Default for unknown strategy
            
            # Ensure fidelity is in valid range
            fidelity = np.clip(fidelity, 0.0, 1.0)
            fidelities.append(fidelity)
            
        except Exception as e:
            print(f"    Error processing sample {i}: {e}")
            fidelities.append(0.5)  # Default fidelity for failed samples
    
    fidelities = np.array(fidelities)
    
    print(f"\nFidelity statistics:")
    print(f"  Mean: {np.mean(fidelities):.4f}")
    print(f"  Std:  {np.std(fidelities):.4f}")
    print(f"  Min:  {np.min(fidelities):.4f}")
    print(f"  Max:  {np.max(fidelities):.4f}")
    
    # Threshold optimization
    thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]
    
    best_gmean = 0
    best_threshold = 0.5
    best_metrics = {}
    threshold_results = []
    
    print(f"\nThreshold optimization:")
    for T in thresholds:
        # Classification rule: Low fidelity (fidelity < T) indicates fraud
        y_pred = (fidelities < T).astype(int)
        m = compute_metrics(y_test, y_pred)
        
        threshold_results.append({
            'threshold': T,
            'metrics': m
        })
        
        print(f"  T={T:.1f}: Acc={m['Accuracy']:.3f} Prec={m['Precision']:.3f} Rec={m['Recall']:.3f} F1={m['F1']:.3f} G-Mean={m['Gmean']:.3f}")
        
        # Track best performance by G-Mean
        if m['Gmean'] > best_gmean:
            best_gmean = m['Gmean']
            best_threshold = T
            best_metrics = m
    
    # Calculate AUC-ROC
    try:
        auc = roc_auc_score(y_test, 1 - fidelities)  # 1-fidelity for anomaly score
    except:
        auc = 0.5
    
    # Ensure best_metrics has all required keys with default values
    if not best_metrics:
        best_metrics = {
            'TN': 0, 'FP': 0, 'FN': 0, 'TP': 0,
            'Accuracy': 0.0, 'Precision': 0.0,
            'Recall': 0.0, 'F1': 0.0, 'Specificity': 0.0, 'Gmean': 0.0
        }
    
    print(f"\nRESULTS SUMMARY:")
    print(f"  AUC-ROC Score: {auc:.4f}")
    print(f"  Best Threshold: {best_threshold} (G-Mean: {best_gmean:.3f})")
    print(f"  Best Performance: Acc={best_metrics.get('Accuracy', 0):.3f}, "
          f"Prec={best_metrics.get('Precision', 0):.3f}, "
          f"Rec={best_metrics.get('Recall', 0):.3f}, "
          f"F1={best_metrics.get('F1', 0):.3f}")
    
    return {
        'experiment_name': experiment_name,
        'strategy': strategy,
        'embedding_type': embedding_type,
        'embedding_config': embedding_config,
        'fidelities': fidelities,
        'auc_roc': auc,
        'best_threshold': best_threshold,
        'best_gmean': best_gmean,
        'best_metrics': best_metrics,
        'threshold_results': threshold_results,
        'training_time': result_data['training_time'],
        'final_loss': result_data['final_loss'],
        'n_qubits': result_data['n_qubits'],
        'total_qubits': result_data['total_qubits']
    }

def evaluate_all_experiments(all_results, y_test):
    """
    Evaluate all experimental results.
    """
    print("STARTING COMPREHENSIVE EVALUATION OF ALL EXPERIMENTS")
    print("="*80)
    
    evaluation_results = {}
    eval_start_time = time.time()
    
    # Evaluate each experiment
    for experiment_name, result_data in all_results.items():
        if 'error' not in result_data:
            eval_result = evaluate_single_experiment(experiment_name, result_data, y_test)
            if eval_result is not None:
                evaluation_results[experiment_name] = eval_result
    
    total_eval_time = time.time() - eval_start_time
    print(f"\n{'='*80}")
    print(f"ALL EVALUATIONS COMPLETED IN {total_eval_time:.1f}s")
    print(f"{'='*80}")
    
    return evaluation_results

print("Comprehensive evaluation functions defined successfully!")
print("  - evaluate_single_experiment: Single experiment evaluator")
print("  - evaluate_all_experiments: All experiments evaluator")
print("  - compute_metrics: Classification metrics with G-Mean")

Comprehensive evaluation functions defined successfully!
  - evaluate_single_experiment: Single experiment evaluator
  - evaluate_all_experiments: All experiments evaluator
  - compute_metrics: Classification metrics with G-Mean


In [10]:
# ==========================================
# Execute Comprehensive Evaluation
# ==========================================

# Evaluate all experiments
final_evaluation_results = evaluate_all_experiments(all_results, y_test)

STARTING COMPREHENSIVE EVALUATION OF ALL EXPERIMENTS

EVALUATING: DIFE_PARALLEL
Strategy: DIFE
Embedding: Parallel Embedding (4D→8Q)
Test data: (190, 4)
Computing reconstruction fidelities for 190 test samples...
  Processed 0/190 samples

Fidelity statistics:
  Mean: 0.5818
  Std:  0.3614
  Min:  0.0000
  Max:  0.9489

Threshold optimization:
  T=0.3: Acc=0.763 Prec=0.981 Rec=0.537 F1=0.694 G-Mean=0.729
  T=0.4: Acc=0.784 Prec=0.966 Rec=0.589 F1=0.732 G-Mean=0.760
  T=0.5: Acc=0.805 Prec=0.939 Rec=0.653 F1=0.770 G-Mean=0.791
  T=0.6: Acc=0.800 Prec=0.880 Rec=0.695 F1=0.776 G-Mean=0.793
  T=0.7: Acc=0.779 Prec=0.805 Rec=0.737 F1=0.769 G-Mean=0.778
  T=0.8: Acc=0.705 Prec=0.689 Rec=0.747 F1=0.717 G-Mean=0.704

RESULTS SUMMARY:
  AUC-ROC Score: 0.8209
  Best Threshold: 0.6 (G-Mean: 0.793)
  Best Performance: Acc=0.800, Prec=0.880, Rec=0.695, F1=0.776

EVALUATING: DIFE_DIRECT
Strategy: DIFE
Embedding: Direct Embedding (8D→8Q)
Test data: (190, 8)
Computing reconstruction fidelities for 190

In [11]:
# ==========================================
# Comprehensive Final Comparison and Analysis
# ==========================================

print(f"\n{'='*120}")
print(f"COMPREHENSIVE COMPARISON: ALL EXPERIMENTS")
print(f"{'='*120}")

if len(final_evaluation_results) == 0:
    print("No experiments were successfully evaluated!")
else:
    # Extract results for analysis
    experiments = ['dife_parallel', 'dife_direct', 'ls_swap_parallel', 'ls_swap_direct']
    successful_experiments = [exp for exp in experiments if exp in final_evaluation_results]
    
    if len(successful_experiments) == 0:
        print("No experiments completed successfully!")
    else:
        print(f"Successfully evaluated {len(successful_experiments)}/4 experiments")
        
        # Performance comparison table
        print(f"\n{'='*120}")
        print(f"PERFORMANCE COMPARISON TABLE")
        print(f"{'='*120}")
        print(f"{'Experiment':<20} {'Strategy':<8} {'Embedding':<12} {'AUC-ROC':<10} {'G-Mean':<10} {'F1':<10} {'Time(s)':<10} {'Qubits':<8}")
        print(f"{'-'*120}")
        
        results_summary = {}
        for exp_name in successful_experiments:
            result = final_evaluation_results[exp_name]
            strategy = result['strategy'].upper()
            embedding_desc = result['embedding_config']['description'].split('(')[0].strip()
            auc = result['auc_roc']
            gmean = result['best_gmean'] 
            f1 = result['best_metrics'].get('F1', 0)
            time_taken = result['training_time']
            qubits = result['total_qubits']
            
            print(f"{exp_name:<20} {strategy:<8} {embedding_desc:<12} {auc:<10.4f} {gmean:<10.3f} {f1:<10.3f} {time_taken:<10.1f} {qubits:<8d}")
            results_summary[exp_name] = {
                'auc': auc, 'gmean': gmean, 'f1': f1, 
                'time': time_taken, 'qubits': qubits,
                'strategy': strategy, 'embedding': embedding_desc
            }
        
        # Strategy comparison (DIFE vs LS-SWAP)
        print(f"\n{'='*120}")
        print(f"STRATEGY ANALYSIS: DIFE vs LS-SWAP")
        print(f"{'='*120}")
        
        dife_experiments = [exp for exp in successful_experiments if exp.startswith('dife')]
        ls_swap_experiments = [exp for exp in successful_experiments if exp.startswith('ls_swap')]
        
        if dife_experiments and ls_swap_experiments:
            # Average performance by strategy
            dife_avg_auc = np.mean([results_summary[exp]['auc'] for exp in dife_experiments])
            ls_swap_avg_auc = np.mean([results_summary[exp]['auc'] for exp in ls_swap_experiments])
            
            dife_avg_gmean = np.mean([results_summary[exp]['gmean'] for exp in dife_experiments])
            ls_swap_avg_gmean = np.mean([results_summary[exp]['gmean'] for exp in ls_swap_experiments])
            
            dife_avg_qubits = np.mean([results_summary[exp]['qubits'] for exp in dife_experiments])
            ls_swap_avg_qubits = np.mean([results_summary[exp]['qubits'] for exp in ls_swap_experiments])
            
            print(f"DIFE Strategy:")
            print(f"  • Average AUC-ROC: {dife_avg_auc:.4f}")
            print(f"  • Average G-Mean: {dife_avg_gmean:.3f}")
            print(f"  • Average Qubits: {dife_avg_qubits:.1f}")
            print(f"  • Experiments: {', '.join(dife_experiments)}")
            
            print(f"\nLS-SWAP Strategy:")
            print(f"  • Average AUC-ROC: {ls_swap_avg_auc:.4f}")
            print(f"  • Average G-Mean: {ls_swap_avg_gmean:.3f}")  
            print(f"  • Average Qubits: {ls_swap_avg_qubits:.1f}")
            print(f"  • Experiments: {', '.join(ls_swap_experiments)}")
            
            strategy_winner = "DIFE" if dife_avg_auc > ls_swap_avg_auc else "LS-SWAP"
            print(f"\n🏆 STRATEGY WINNER: {strategy_winner}")
        
        # Embedding comparison (Parallel vs Direct)
        print(f"\n{'='*120}")
        print(f"EMBEDDING ANALYSIS: Parallel vs Direct")
        print(f"{'='*120}")
        
        parallel_experiments = [exp for exp in successful_experiments if exp.endswith('parallel')]
        direct_experiments = [exp for exp in successful_experiments if exp.endswith('direct')]
        
        if parallel_experiments and direct_experiments:
            # Average performance by embedding
            parallel_avg_auc = np.mean([results_summary[exp]['auc'] for exp in parallel_experiments])
            direct_avg_auc = np.mean([results_summary[exp]['auc'] for exp in direct_experiments])
            
            parallel_avg_gmean = np.mean([results_summary[exp]['gmean'] for exp in parallel_experiments])
            direct_avg_gmean = np.mean([results_summary[exp]['gmean'] for exp in direct_experiments])
            
            print(f"Parallel Embedding (4D→8Q):")
            print(f"  • Average AUC-ROC: {parallel_avg_auc:.4f}")
            print(f"  • Average G-Mean: {parallel_avg_gmean:.3f}")
            print(f"  • Data efficiency: Replicates 4D features across 8 qubits")
            print(f"  • Experiments: {', '.join(parallel_experiments)}")
            
            print(f"\nDirect Embedding (8D→8Q):")
            print(f"  • Average AUC-ROC: {direct_avg_auc:.4f}")
            print(f"  • Average G-Mean: {direct_avg_gmean:.3f}")
            print(f"  • Data efficiency: Maps 8D features directly to 8 qubits")
            print(f"  • Experiments: {', '.join(direct_experiments)}")
            
            embedding_winner = "Parallel" if parallel_avg_auc > direct_avg_auc else "Direct"
            print(f"\n🏆 EMBEDDING WINNER: {embedding_winner}")
        
        # Overall best experiment
        print(f"\n{'='*120}")
        print(f"OVERALL ANALYSIS")
        print(f"{'='*120}")
        
        # Find best experiment by AUC-ROC
        best_exp = max(successful_experiments, key=lambda exp: results_summary[exp]['auc'])
        best_result = results_summary[best_exp]
        
        print(f"🥇 OVERALL CHAMPION: {best_exp.upper()}")
        print(f"   Strategy: {best_result['strategy']}")
        print(f"   Embedding: {best_result['embedding']}")
        print(f"   AUC-ROC: {best_result['auc']:.4f}")
        print(f"   G-Mean: {best_result['gmean']:.3f}")
        print(f"   F1-Score: {best_result['f1']:.3f}")
        print(f"   Training Time: {best_result['time']:.1f}s")
        print(f"   Total Qubits: {best_result['qubits']}")
        
        # Resource efficiency analysis
        print(f"\n📊 RESOURCE EFFICIENCY ANALYSIS:")
        efficiency_scores = {}
        for exp_name in successful_experiments:
            result = results_summary[exp_name]
            efficiency = result['auc'] / result['qubits']  # AUC per qubit
            efficiency_scores[exp_name] = efficiency
        
        most_efficient_exp = max(efficiency_scores.keys(), key=lambda exp: efficiency_scores[exp])
        print(f"   Most Efficient: {most_efficient_exp.upper()}")
        print(f"   Efficiency Score: {efficiency_scores[most_efficient_exp]:.6f} AUC/qubit")
        
        print(f"\n   Efficiency Rankings:")
        for exp, efficiency in sorted(efficiency_scores.items(), key=lambda x: x[1], reverse=True):
            result = results_summary[exp]
            print(f"   {exp:<20}: {efficiency:.6f} ({result['auc']:.4f} AUC / {result['qubits']} qubits)")
        
        # Key insights
        print(f"\n{'='*120}")
        print(f"KEY INSIGHTS AND RECOMMENDATIONS")
        print(f"{'='*120}")
        
        print(f"\n🔍 STRATEGY INSIGHTS:")
        if len(dife_experiments) > 0 and len(ls_swap_experiments) > 0:
            if dife_avg_auc > ls_swap_avg_auc:
                print(f"   • DIFE outperforms LS-SWAP on average ({dife_avg_auc:.4f} vs {ls_swap_avg_auc:.4f})")
                print(f"   • DIFE is more resource-efficient (fewer qubits)")
            else:
                print(f"   • LS-SWAP outperforms DIFE on average ({ls_swap_avg_auc:.4f} vs {dife_avg_auc:.4f})")
                print(f"   • LS-SWAP uses more resources but may provide better accuracy")
        
        print(f"\n🔧 EMBEDDING INSIGHTS:")
        if len(parallel_experiments) > 0 and len(direct_experiments) > 0:
            if parallel_avg_auc > direct_avg_auc:
                print(f"   • Parallel embedding is more effective ({parallel_avg_auc:.4f} vs {direct_avg_auc:.4f})")
                print(f"   • Data replication across qubits improves feature representation")
            else:
                print(f"   • Direct embedding is more effective ({direct_avg_auc:.4f} vs {parallel_avg_auc:.4f})")
                print(f"   • Higher dimensional input (8D vs 4D) provides better information")
        
        print(f"\n💡 PRACTICAL RECOMMENDATIONS:")
        if best_result['auc'] >= 0.85:
            print(f"   🟢 EXCELLENT: {best_exp} achieves high fraud detection performance")
        elif best_result['auc'] >= 0.75:
            print(f"   🟡 GOOD: {best_exp} provides reasonable fraud detection capability")
        else:
            print(f"   🔴 MODERATE: Best performance may need improvement for production use")
        
        print(f"   • For resource-constrained environments: Choose {most_efficient_exp}")
        print(f"   • For maximum accuracy: Choose {best_exp}")
        print(f"   • Trade-off consideration: Balance between performance and qubit requirements")
        
        # Save results for further analysis
        comprehensive_results = {
            'evaluation_results': final_evaluation_results,
            'summary': results_summary,
            'best_experiment': best_exp,
            'most_efficient': most_efficient_exp,
            'strategy_comparison': {
                'dife_avg_auc': dife_avg_auc if 'dife_avg_auc' in locals() else None,
                'ls_swap_avg_auc': ls_swap_avg_auc if 'ls_swap_avg_auc' in locals() else None
            },
            'embedding_comparison': {
                'parallel_avg_auc': parallel_avg_auc if 'parallel_avg_auc' in locals() else None,
                'direct_avg_auc': direct_avg_auc if 'direct_avg_auc' in locals() else None
            }
        }

print(f"\n{'='*120}")
print(f"COMPREHENSIVE ANALYSIS COMPLETE")
print(f"{'='*120}")
print(f"✅ All experiments successfully compared!")
print(f"📊 Results available in 'comprehensive_results' variable")
print(f"🎯 Ready for production deployment considerations!")


COMPREHENSIVE COMPARISON: ALL EXPERIMENTS
Successfully evaluated 4/4 experiments

PERFORMANCE COMPARISON TABLE
Experiment           Strategy Embedding    AUC-ROC    G-Mean     F1         Time(s)    Qubits  
------------------------------------------------------------------------------------------------------------------------
dife_parallel        DIFE     Parallel Embedding 0.8209     0.793      0.776      2346.6     8       
dife_direct          DIFE     Direct Embedding 0.8160     0.799      0.791      2338.2     8       
ls_swap_parallel     LS_SWAP  Parallel Embedding 0.9105     0.877      0.886      1993.3     11      
ls_swap_direct       LS_SWAP  Direct Embedding 0.8874     0.847      0.850      2169.7     11      

STRATEGY ANALYSIS: DIFE vs LS-SWAP
DIFE Strategy:
  • Average AUC-ROC: 0.8184
  • Average G-Mean: 0.796
  • Average Qubits: 8.0
  • Experiments: dife_parallel, dife_direct

LS-SWAP Strategy:
  • Average AUC-ROC: 0.8989
  • Average G-Mean: 0.862
  • Average Qubits: 1

In [12]:
# ==========================================
# Detailed Results Analysis
# ==========================================

print("DETAILED PERFORMANCE METRICS FOR ALL EXPERIMENTS")
print("="*80)

if 'comprehensive_results' in locals() and comprehensive_results:
    evaluation_results = comprehensive_results['evaluation_results']
    
    for exp_name, result in evaluation_results.items():
        print(f"\n{exp_name.upper()} DETAILED METRICS:")
        print(f"  Strategy: {result['strategy'].upper()}")
        print(f"  Embedding: {result['embedding_config']['description']}")
        
        metrics = result['best_metrics']
        print(f"  Accuracy:    {metrics.get('Accuracy', 0):.4f}")
        print(f"  Precision:   {metrics.get('Precision', 0):.4f}")
        print(f"  Recall:      {metrics.get('Recall', 0):.4f}")
        print(f"  F1-Score:    {metrics.get('F1', 0):.4f}")
        print(f"  Specificity: {metrics.get('Specificity', 0):.4f}")
        print(f"  G-Mean:      {metrics.get('Gmean', 0):.4f}")
        print(f"  AUC-ROC:     {result['auc_roc']:.4f}")
        print(f"  Best Threshold: {result['best_threshold']:.1f}")
        print(f"  Training Time: {result['training_time']:.1f}s")
        print(f"  Total Qubits: {result['total_qubits']}")
        
        # Confusion Matrix
        print(f"  Confusion Matrix:")
        print(f"    True Negatives (TN):  {metrics.get('TN', 0)}")
        print(f"    False Positives (FP): {metrics.get('FP', 0)}")
        print(f"    False Negatives (FN): {metrics.get('FN', 0)}")
        print(f"    True Positives (TP):  {metrics.get('TP', 0)}")
else:
    print("Comprehensive results not available. Please run all experiments first.")

print("\n" + "="*80)
print("ANALYSIS COMPLETE - All experiments evaluated and compared!")
print("="*80)

DETAILED PERFORMANCE METRICS FOR ALL EXPERIMENTS

DIFE_PARALLEL DETAILED METRICS:
  Strategy: DIFE
  Embedding: Parallel Embedding (4D→8Q)
  Accuracy:    0.8000
  Precision:   0.8800
  Recall:      0.6947
  F1-Score:    0.7765
  Specificity: 0.9053
  G-Mean:      0.7930
  AUC-ROC:     0.8209
  Best Threshold: 0.6
  Training Time: 2346.6s
  Total Qubits: 8
  Confusion Matrix:
    True Negatives (TN):  86
    False Positives (FP): 9
    False Negatives (FN): 29
    True Positives (TP):  66

DIFE_DIRECT DETAILED METRICS:
  Strategy: DIFE
  Embedding: Direct Embedding (8D→8Q)
  Accuracy:    0.8000
  Precision:   0.8276
  Recall:      0.7579
  F1-Score:    0.7912
  Specificity: 0.8421
  G-Mean:      0.7989
  AUC-ROC:     0.8160
  Best Threshold: 0.3
  Training Time: 2338.2s
  Total Qubits: 8
  Confusion Matrix:
    True Negatives (TN):  80
    False Positives (FP): 15
    False Negatives (FN): 23
    True Positives (TP):  72

LS_SWAP_PARALLEL DETAILED METRICS:
  Strategy: LS_SWAP
  Embeddin